## 선택 · 심화 문제 1. 두 vocabulary 정책의 길이·UNK 비교

### 문제 배경

큰 vocabulary는 긴 조각을 담아 sequence를 줄일 수 있지만 embedding parameter를 늘립니다. 두 toy vocabulary를 같은 corpus에 적용해 평균 token 수, UNK 수, embedding parameter 수를 비교합니다.

### 시작 코드

```python
corpus = [["인공지능", "연구소", "출범"], ["인공지능", "연구"], ["미등록", "출범"]]
small_vocab = vocab
large_vocab = {**vocab, "인공지능": 9, "인공지능연구소": 10, "미등록": 11}

def compare_vocabularies(corpus, vocabularies, embedding_dim=8):
    raise NotImplementedError
```

### 수행 요구사항

1. 각 vocabulary로 모든 문장을 encode하세요.
2. Special token을 제외한 평균 token 수와 UNK 수를 계산하세요.
3. `len(vocab)*embedding_dim`으로 embedding parameter 수를 계산하세요.
4. 더 큰 vocabulary가 무조건 더 좋은 것은 아닌 이유를 설명하세요.

### 제출 결과

- `small/large` 비교표
- 길이 감소와 parameter 증가 해석
- `심화 문제 1 자동 검증: PASS`

### 자동 검증

```python
table = compare_vocabularies(corpus, {"small": small_vocab, "large": large_vocab})
assert table["large"]["mean_tokens"] < table["small"]["mean_tokens"]
assert table["large"]["unk_count"] < table["small"]["unk_count"]
assert table["large"]["embedding_parameters"] > table["small"]["embedding_parameters"]
print("심화 문제 1 자동 검증: PASS")
```

    ```
    
**상세 해설** · 큰 vocabulary가 이 작은 corpus에서는 길이와 UNK를 줄였지만 embedding parameter는 늘었습니다. 실제 tokenizer 선택은 학습 corpus 대표성, downstream 성능, 모델 checkpoint 호환성까지 평가해야 합니다.
    
  **자주 하는 실수**
    
    - `[CLS]/[SEP]`를 content token 길이에 포함해 정책 차이를 흐립니다.
    - Vocabulary가 크면 OOV가 항상 0이라고 일반화합니다.
    - Embedding뿐 아니라 LM head가 vocabulary 크기의 영향을 받을 수 있다는 점을 놓칩니다.

In [1]:
vocab = {"[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3,
         "인공": 4, "##지능": 5, "연구": 6, "##소": 7, "출범": 8}

def split_word(word, vocab):
    pieces, start = [], 0
    while start < len(word):
        found = None
        # 현재 위치에서 가장 긴 등록 조각이 먼저 선택되도록 뒤에서부터 탐색합니다.
        for end in range(len(word), start, -1):
            token = word[start:end] if start == 0 else "##" + word[start:end]
            if token in vocab:
                found = (token, end); break
        if found is None:
            return ["[UNK]"]
        token, start = found; pieces.append(token)
    return pieces

def encode_words(words, vocab):
    tokens = ["[CLS]"]
    for word in words: tokens.extend(split_word(word, vocab))
    tokens.append("[SEP]")
    return tokens, [vocab[token] for token in tokens]

corpus = [["인공지능", "연구소", "출범"], ["인공지능", "연구"], ["미등록", "출범"]]
small_vocab = vocab
large_vocab = {**vocab, "인공지능": 9, "인공지능연구소": 10, "미등록": 11}

def compare_vocabularies(corpus, vocabularies, embedding_dim=8):
    table = {}
    for name, current_vocab in vocabularies.items():
        # 같은 corpus를 두 vocabulary로 인코딩해야 비교 조건이 유지됩니다.
        encoded = [encode_words(words, current_vocab)[0] for words in corpus]
        content_lengths = [len(tokens) - 2 for tokens in encoded]  # CLS/SEP 제외
        table[name] = {
            "mean_tokens": sum(content_lengths) / len(content_lengths),
            "unk_count": sum(tokens.count("[UNK]") for tokens in encoded),
            "embedding_parameters": len(current_vocab) * embedding_dim,
        }
    return table

table = compare_vocabularies(corpus, {"small": small_vocab, "large": large_vocab})
print(table)
assert table["large"]["mean_tokens"] < table["small"]["mean_tokens"]
assert table["large"]["unk_count"] < table["small"]["unk_count"]
assert table["large"]["embedding_parameters"] > table["small"]["embedding_parameters"]
print("심화 문제 1 자동 검증: PASS")

{'small': {'mean_tokens': 3.3333333333333335, 'unk_count': 1, 'embedding_parameters': 72}, 'large': {'mean_tokens': 2.6666666666666665, 'unk_count': 0, 'embedding_parameters': 96}}
심화 문제 1 자동 검증: PASS
